<font size=10>**NETWORK**</font> <a class="anchor" id='title'></a> 

**Bachelor's in Data Science - NOVA IMS (25/26)**

<font color='#BFD72' size=5>**RESEARCH QUESTION**: </font><font size=5>*Which companies have a dominant position within specific municipalities?*</font> 

**Data**: 
- [*Portal BASE*](https://www.base.gov.pt/Base4/pt/pesquisa/?type=contratos&texto=&adjudicante=&adjudicataria=&tipo=2&tipocontrato=0&cpv=&aqinfo=&desdeprazoexecucao=&ateprazoexecucao=&sel_price=price_11&desdeprecocontrato=&ateprecocontrato=&desdeprecoefectivo=&ateprecoefectivo=&sel_date=date_11&desdedatacontrato=2023-01-01&atedatacontrato=2026-03-31&desdedatapublicacao=&atedatapublicacao=&desdedatafecho=&atedatafecho=&pais=0&distrito=0&concelho=0)

- [*Treated Datasets*](https://dados.gov.pt/pt/datasets/contratos-publicos-portal-base-impic-contratos-de-2012-a-2026/#/resources)

**Group B**
- Beatriz Marques 20231605
- Maria Inês Santos 20231630
- Luís Soeiro 20211536
- Rodrigo Silva 20231602

<font color='#BFD72' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a>  
- [1. Imports](#1)  
- [2. Data Integration](#2)  
- [3. First Network](#3)

# <font color='#BFD72F' size=6>**1. Imports**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

In [1]:
import warnings
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')

Failed to read module file 'C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\functools.py' for module 'functools': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Users\wwwnj\OneDrive - NOVAIMS\Desktop\Network Analysis\.venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\wwwnj\OneDrive - NOVAIMS\Desktop\Network Analysis\.venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\importlib\__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap

In [2]:
import sys
import os

# Get the absolute path of the source_code folder
source_code_path = os.path.abspath('../source')

# Add the source_code folder to sys.path
if source_code_path not in sys.path:
    sys.path.append(source_code_path)

In [3]:
import subprocess, sys, importlib
import pandas as pd
import plotly.express as px
import re
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go

try:
    import openpyxl
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
    importlib.invalidate_caches()
    import openpyxl
    
import os
import shutil

# <font color='#BFD72F' size=6>**2. Data Integration**</font> <a class="anchor" id="2"></a>
  
[Back to TOC](#toc)

In [4]:
data = pd.read_csv('../data/preprocessed_data.csv')
# data.head()
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 690 entries, 0 to 689
Data columns (total 22 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   idcontrato                   690 non-null    int64  
 1   tipoContrato                 690 non-null    str    
 2   tipoFimContrato              690 non-null    str    
 3   CPV                          690 non-null    str    
 4   adjudicante                  690 non-null    str    
 5   adjudicatarios               690 non-null    str    
 6   concorrentes                 482 non-null    str    
 7   precoBaseProcedimento        690 non-null    float64
 8   precoContratual              690 non-null    float64
 9   PrecoTotalEfetivo            690 non-null    float64
 10  dataDecisaoAdjudicacao       690 non-null    str    
 11  dataCelebracaoContrato       690 non-null    str    
 12  dataPublicacao               690 non-null    str    
 13  dataFechoContrato            69

In [5]:
data["dataPublicacao"] = pd.to_datetime(data["dataPublicacao"], errors='coerce')
data["dataCelebracaoContrato"] = pd.to_datetime(data["dataCelebracaoContrato"], errors='coerce')
data["dataDecisaoAdjudicacao"] = pd.to_datetime(data["dataDecisaoAdjudicacao"], errors='coerce')
data["dataFechoContrato"] = pd.to_datetime(data["dataFechoContrato"], errors='coerce')

# <font color='#BFD72F' size=6>**3. The Network**</font> <a class="anchor" id="3"></a>
  
[Back to TOC](#toc)

## <font size=5>**3.1 Creating It**</font> <a class="anchor" id="3.1"></a>
  
[Back to TOC](#toc)

In [6]:
################
### BEA CODE ###
################

# # If your data is not a DataFrame, fix it
# if isinstance(data, dict):
#     data = pd.DataFrame(data)

# # Create directed graph
# G = nx.DiGraph()

# # Build graph
# for _, row in data.iterrows():
#     adjudicante = row['adjudicante']
#     adjudicatarios = row['adjudicatarios']
#     preco = row['precoContratual']
#     concorrentes = row.get('concorrentes', 0)

#     # Skip missing
#     if pd.isna(adjudicante) or pd.isna(adjudicatarios):
#         continue
#     if pd.isna(preco) or preco <= 0:
#         continue

#     log_price = np.log(preco)

#     # Handle concorrentes safely
#     if pd.isna(concorrentes):
#         concorrentes = 0

#     # Add/update nodes
#     if adjudicante not in G:
#         G.add_node(adjudicante, node_type='adjudicante')
#     else:
#         if G.nodes[adjudicante].get('node_type') == 'adjudicatario':
#             G.nodes[adjudicante]['node_type'] = 'both'

#     if adjudicatarios not in G:
#         G.add_node(adjudicatarios, node_type='adjudicatario')
#     else:
#         if G.nodes[adjudicatarios].get('node_type') == 'adjudicante':
#             G.nodes[adjudicatarios]['node_type'] = 'both'

#     # Add/update edge
#     if G.has_edge(adjudicante, adjudicatarios):
#         edge = G[adjudicante][adjudicatarios]
#         edge['weight'] += log_price
#         edge['contracts'] += 1
#         edge_conc = pd.to_numeric(edge.get('nr_concorrentes', 0), errors='coerce')
#         row_conc = pd.to_numeric(concorrentes, errors='coerce')
#         edge['nr_concorrentes'] = int(0 if pd.isna(edge_conc) else edge_conc) + int(0 if pd.isna(row_conc) else row_conc)
#     else:
#         G.add_edge(
#             adjudicante,
#             adjudicatarios,
#             weight=log_price,
#             contracts=1,
#             nr_concorrentes=concorrentes,
#             city=row.get('city', None)
#         )

In [7]:
def compute_weight(prices, mode="log_sum"):

    prices = np.array(prices)

    if mode == "sum_price":
        return prices.sum()

    elif mode == "avg_price":
        return prices.mean()

    elif mode == "log_sum":
        return np.log(prices).sum()

    elif mode == "log_mean":
        return np.log(prices).mean()

    else:
        raise ValueError(
            f"Unknown mode: {mode}. "
            f"Valid modes: sum_price, avg_price, log_sum, log_mean"
        )

In [8]:
def _ensure_dataframe(data):
    if isinstance(data, pd.DataFrame):
        return data.copy()

    if isinstance(data, dict):
        # CASE 1: dict of lists (valid dataframe)
        try:
            df = pd.DataFrame(data)
            return df
        except Exception:
            pass

        # CASE 2: dict of scalars → wrap into list
        return pd.DataFrame([data])

    raise TypeError(f"Unsupported input type: {type(data)}")

In [9]:
def build_contract_network(data, weight_mode="log_sum") -> nx.DiGraph:
    
    df = _ensure_dataframe(data)

    # --- CLEAN COLUMNS ---
    df = df.rename(columns={
        'precoContratual': 'price',
        'adjudicante': 'source',
        'adjudicatarios': 'target'
    })

    # --- BASIC CLEANING ---
    df = df.dropna(subset=['source', 'target', 'price'])
    df = df[df['price'] > 0]

    # --- CONCORRENTES ---
    if 'nr_concorrentes' in df.columns:
        df['concorrentes'] = df['nr_concorrentes']
    else:
        df['concorrentes'] = pd.to_numeric(df.get('concorrentes', 0), errors='coerce').fillna(0)

    # --- EDGE AGGREGATION ---
    edge_df = (
        df.groupby(['source', 'target'], as_index=False)
        .agg(
            total_price=('price', 'sum'),
            nr_concorrentes=('concorrentes', 'sum'),
            contracts=('price', 'count'),
            price_series=('price', list),
            idcontrato=('idcontrato', list),
            tipoContrato=('tipoContrato', list),
            tipoFimContrato=('tipoFimContrato', list),
            CPV=('CPV', list),
            precoBaseProcedimento=('precoBaseProcedimento', list),
            precoContratual=('price', list),
            PrecoTotalEfetivo=('PrecoTotalEfetivo', list),
            dataDecisaoAdjudicacao=('dataDecisaoAdjudicacao', list),
            dataCelebracaoContrato=('dataCelebracaoContrato', list),
            dataPublicacao=('dataPublicacao', list),
            dataFechoContrato=('dataFechoContrato', list),
            nr_concorrentes_list=('nr_concorrentes', list),
            contribuinte_adjudicante=('contribuinte_adjudicante', list),
            contribuinte_adjudicatarios=('contribuinte_adjudicatarios', list),
            city=('city', list),
            cpv_prefix=('cpv_prefix', list),
            agg_cpv=('agg_cpv', list)
        )
    )

    # --- WEIGHT STRATEGY ---
    edge_df['weight'] = edge_df['price_series'].apply(
        lambda x: compute_weight(x, mode=weight_mode)
    )

    edge_df = edge_df.drop(columns=['price_series'])

    # --- BUILD GRAPH ---
    G = nx.DiGraph()

    for row in edge_df.itertuples(index=False):
        G.add_edge(
            row.source,
            row.target,
            weight=row.weight,
            total_price=row.total_price,
            nr_concorrentes=row.nr_concorrentes,
            contracts=row.contracts,
            weight_mode=weight_mode,
            idcontrato=row.idcontrato,
            tipoContrato=row.tipoContrato,
            tipoFimContrato=row.tipoFimContrato,
            CPV=row.CPV,
            precoBaseProcedimento=row.precoBaseProcedimento,
            precoContratual=row.precoContratual,
            PrecoTotalEfetivo=row.PrecoTotalEfetivo,
            dataDecisaoAdjudicacao=row.dataDecisaoAdjudicacao,
            dataCelebracaoContrato=row.dataCelebracaoContrato,
            dataPublicacao=row.dataPublicacao,
            dataFechoContrato=row.dataFechoContrato,
            nr_concorrentes_list=row.nr_concorrentes_list,
            contribuinte_adjudicante=row.contribuinte_adjudicante,
            contribuinte_adjudicatarios=row.contribuinte_adjudicatarios,
            city=row.city,
            cpv_prefix=row.cpv_prefix,
            agg_cpv=row.agg_cpv
        )

    # --- NODE TYPES ---
    sources = set(edge_df['source'])
    targets = set(edge_df['target'])

    for node in G.nodes():
        if node in sources and node in targets:
            G.nodes[node]['node_type'] = 'both'
        elif node in sources:
            G.nodes[node]['node_type'] = 'adjudicante'
        else:
            G.nodes[node]['node_type'] = 'adjudicatario'

    return G

# -------------------------------
# VISUAL ATTRIBUTES (UNIFIED)
# -------------------------------
def add_visual_attributes(
    G,
    thickness_mode="linear",
    size_mode="linear",
    scale_min=1,
    scale_max=5,
    suffix=""
):
    conc = np.array([d['nr_concorrentes'] for _, _, d in G.edges(data=True)])
    weight = np.array([d['weight'] for _, _, d in G.edges(data=True)])

    def normalize(x):
        if len(x) == 0:
            return x

        range_ = np.ptp(x)  # max - min safely

        if range_ == 0:
            return np.zeros_like(x)

        return (x - np.min(x)) / (range_ + 1e-9)

    # --- TRANSFORM ---
    if thickness_mode == "log":
        conc = np.log1p(conc)

    if size_mode == "log":
        weight = np.log1p(weight)

    conc_norm = normalize(conc)
    weight_norm = normalize(weight)

    def scale(x):
        return scale_min + (scale_max - scale_min) * x

    # --- ASSIGN ---
    for i, (u, v, d) in enumerate(G.edges(data=True)):
        d[f'edge_thickness{suffix}'] = scale(conc_norm[i])
        d[f'edge_size{suffix}'] = scale(weight_norm[i])

    return G


# -------------------------------
# RUN PIPELINE
# -------------------------------
G = build_contract_network(data, weight_mode="log_sum")

# Linear scaling
G_linear = add_visual_attributes(
    G.copy(),
    thickness_mode="linear",
    size_mode="linear",
    suffix="_linear"
)

# Log scaling
G_log = add_visual_attributes(
    G.copy(),
    thickness_mode="log",
    size_mode="log",
    suffix="_log"
)

In [10]:
weight_modes = ["sum_price", "avg_price", "log_sum", "log_mean"]

graphs = {
    mode: build_contract_network(data, weight_mode=mode)
    for mode in weight_modes
}

In [11]:
for mode, G in graphs.items():
    volume = {
        node: sum(d["total_price"] for _, _, d in G.edges(node, data=True))
        for node in G.nodes()
    }

    degree = dict(G.degree())

    corr = pd.Series(degree).corr(pd.Series(volume))

    print(f"{mode:10s} → degree-volume correlation: {corr:.3f}")

sum_price  → degree-volume correlation: 0.627
avg_price  → degree-volume correlation: 0.627
log_sum    → degree-volume correlation: 0.627
log_mean   → degree-volume correlation: 0.627


In [12]:
for mode, G in graphs.items():
    top = sorted(G.degree(), key=lambda x: x[1], reverse=True)[:3]

    print(f"\nMode: {mode}")
    for node, deg in top:
        print(f"  {node}: {deg}")


Mode: sum_price
  Guarda Nacional Republicana: 66
  Secretaria-Geral do Ministério da Administração Interna: 28
  Serviços Municipalizados de Água e Saneamento de Sintra: 19

Mode: avg_price
  Guarda Nacional Republicana: 66
  Secretaria-Geral do Ministério da Administração Interna: 28
  Serviços Municipalizados de Água e Saneamento de Sintra: 19

Mode: log_sum
  Guarda Nacional Republicana: 66
  Secretaria-Geral do Ministério da Administração Interna: 28
  Serviços Municipalizados de Água e Saneamento de Sintra: 19

Mode: log_mean
  Guarda Nacional Republicana: 66
  Secretaria-Geral do Ministério da Administração Interna: 28
  Serviços Municipalizados de Água e Saneamento de Sintra: 19


In [13]:
rankings = {}

for mode, G in graphs.items():

    weighted_degree = {
        node: sum(d["weight"] for _, _, d in G.edges(node, data=True))
        for node in G.nodes()
    }

    # convert to ranking (higher = better rank)
    ranked = pd.Series(weighted_degree).rank(ascending=False)

    rankings[mode] = ranked

sum_price → total contract value emphasis

avg_price → typical contract size

log_sum → interaction-frequency + heavy-tail compression

log_mean → normalized interaction intensity

In [14]:
rank_df = pd.DataFrame(rankings)
rank_df.head()

,sum_price,avg_price,log_sum,log_mean
"ACSS-Administração Central do Sistema de Saúde,IP",99.0,97.0,61.0,57.0
"CLARANET II SOLUTIONS, S.A.",390.0,390.0,390.0,390.0
Primavera – Business Software Solutions S.A,390.0,390.0,390.0,390.0
"Timestamp - Sistemas de Informação, S.A.",390.0,390.0,390.0,390.0
ADENE - Agência para a Energia,87.0,85.0,82.0,80.0


In [15]:
rank_df["rank_std"] = rank_df.std(axis=1)
rank_df["rank_mean"] = rank_df.mean(axis=1)

In [16]:
# Structurally important regardless of weight strategy

stable_nodes = rank_df.sort_values("rank_std").head(10)
stable_nodes

,sum_price,avg_price,log_sum,log_mean,rank_std,rank_mean
dragondisplay unipessoal lda,390.0,390.0,390.0,390.0,0.0,312.0
"CLARANET II SOLUTIONS, S.A.",390.0,390.0,390.0,390.0,0.0,312.0
Primavera – Business Software Solutions S.A,390.0,390.0,390.0,390.0,0.0,312.0
"Timestamp - Sistemas de Informação, S.A.",390.0,390.0,390.0,390.0,0.0,312.0
NÓS COMUNICAÇÕES S.A.,390.0,390.0,390.0,390.0,0.0,312.0
Digibéria Information Technologies S.A.,390.0,390.0,390.0,390.0,0.0,312.0
"Páginas aos Blocos, Lda.",390.0,390.0,390.0,390.0,0.0,312.0
PH Informática,390.0,390.0,390.0,390.0,0.0,312.0
"MULTIMAC HITO INNOVATION, S.A.",390.0,390.0,390.0,390.0,0.0,312.0
"PREMIUM GREEN MAIL, LDA",390.0,390.0,390.0,390.0,0.0,312.0


<font color ='red'>These nodes:
- have identical ranks across ALL weighting schemes
- are structurally invariant in your network

They are topological anchors, not sensitive to how we measure money or interaction.

We see identical rank values like 5594.5, this suggests many tied ranks. So they are stable and saturated centrality nodes. This means that ranking resolution is too coarse and many nodes are indistinguishable under this metric.

In [17]:
# Importende depends heavily on how we measure importance

unstable_nodes = rank_df.sort_values("rank_std", ascending=False).head(10)
unstable_nodes

,sum_price,avg_price,log_sum,log_mean,rank_std,rank_mean
Área Metropolitana de Lisboa,54.0,124.0,10.0,130.0,57.766772,75.153354
"Instituto do Emprego e Formação Profissional, IP",8.0,7.0,88.0,87.0,46.191630,47.238326
"Imprensa Nacional - Casa da Moeda, S. A.",115.0,113.0,47.0,43.0,39.878984,71.575797
"Instituto Nacional de Saúde Doutor Ricardo Jorge, I. P.",91.0,89.0,26.0,23.0,37.845079,53.369016
SUCH | Serviço de Utilização Comum dos Hospitais,63.0,111.0,23.0,81.0,36.783148,62.956630
Município da Lourinhã,110.0,115.0,44.0,56.0,36.472592,72.294518
"Cascais Dinâmica - Gestão de Economia, Turismo e Empreendorismo, E. M., S. A.",29.0,28.0,89.0,88.0,34.645827,53.729165
"Agência Nacional para a Qualificação e o Ensino Profissional, I. P.",77.0,73.0,16.0,15.0,34.393556,43.078711
Faculdade de Letras da Universidade de Lisboa,98.0,96.0,42.0,39.0,32.653484,61.530697
Instituto Superior de Economia e Gestão,67.0,64.0,11.0,8.0,32.377976,36.475595


<font color='red'>These nodes change position dramatically depending on weighting scheme. Their importance depends on what we consider "importance". 

Pattern:
1. Mixed contract sizes
some large contracts
many small ones
2. irregular interaction structure
sporadic procurement behavior
3. heterogeneity in edges

sum_price -> rewards big contracts
avg_price -> smooths variability
log_sum -> rewards frequency
log_mean -> penalizes extreme dispersion

In [18]:
# # Visualize the graph
# # Layout (can take time for big graphs)
# pos = nx.spring_layout(G, k=0.15, iterations=20, seed=42)

# # Extract edge attributes
# weights = [G[u][v]['weight'] for u, v in G.edges()]
# widths = [G[u][v]['nr_concorrentes'] for u, v in G.edges()]

# # Avoid division by zero
# if len(widths) > 0 and max(widths) > 0:
#     widths = [w / max(widths) * 5 for w in widths]
# else:
#     widths = [1 for _ in widths]

# if len(weights) > 0 and max(weights) > 0:
#     weights_norm = [w / max(weights) for w in weights]
# else:
#     weights_norm = weights

# # Draw
# plt.figure(figsize=(12, 10))

# nx.draw_networkx_nodes(G, pos, node_size=50)

# nx.draw_networkx_edges(
#     G,
#     pos,
#     width=widths,                 # thickness = concorrentes
#     edge_color=weights_norm,      # color = log(price)
#     edge_cmap=plt.cm.Blues
# )

# plt.title("Network: Adjudicante → Adjudicatário")
# plt.axis('off')
# plt.show()

## <font size=5>**3.2 General Insights**</font> <a class="anchor" id="3.2"></a>
  
[Back to TOC](#toc)

In [19]:
# Number of nodes
print(f"Number of nodes: {G.number_of_nodes()}")
print(f"\t-> Number of adjudicantes: {sum(1 for _, attr in G.nodes(data=True) if attr['node_type'] in ['adjudicante', 'both'])}, i.e., {sum(1 for _, attr in G.nodes(data=True) if attr['node_type'] in ['adjudicante', 'both']) / G.number_of_nodes() * 100:.2f}%")
print(f"\t-> Number of adjudicatarios: {sum(1 for _, attr in G.nodes(data=True) if attr['node_type'] in ['adjudicatario', 'both'])}, i.e., {sum(1 for _, attr in G.nodes(data=True) if attr['node_type'] in ['adjudicatario', 'both']) / G.number_of_nodes() * 100:.2f}%")

# Number of edges
print(f"\nNumber of edges: {G.number_of_edges()}")

Number of nodes: 641
	-> Number of adjudicantes: 138, i.e., 21.53%
	-> Number of adjudicatarios: 504, i.e., 78.63%

Number of edges: 589


## <font size=5>**3.3 Rank All Entities By Degree (Number of Contracts)**</font> <a class="anchor" id="3.3"></a>
  
[Back to TOC](#toc)

In [20]:
def entity_type(node):
    node_type = G.nodes[node].get("node_type", "unknown")
    if node_type == "both":
        return "adjudicante and adjudicatario"
    return node_type

# Highest and lowest in-degree
highest_in = max(G.in_degree(), key=lambda x: x[1])
lowest_in = min(G.in_degree(), key=lambda x: x[1])

# Highest and lowest out-degree
highest_out = max(G.out_degree(), key=lambda x: x[1])
lowest_out = min(G.out_degree(), key=lambda x: x[1])

# Average in-degree and out-degree
in_degrees = [d for _, d in G.in_degree()]
out_degrees = [d for _, d in G.out_degree()]

avg_in = sum(in_degrees) / len(in_degrees)
avg_out = sum(out_degrees) / len(out_degrees)

print(f"Highest in-degree: {highest_in} -> {entity_type(highest_in[0])}")
print(f"Lowest out-degree: {lowest_out} -> {entity_type(lowest_out[0])}")

print(f"\nHighest out-degree: {highest_out} -> {entity_type(highest_out[0])}")
print(f"Lowest in-degree: {lowest_in} -> {entity_type(lowest_in[0])}")

print(f"\nAverage in-degree: {avg_in:.2f}")
print(f"Average out-degree: {avg_out:.2f}")

Highest in-degree: ('CLARANET II SOLUTIONS, S.A.', 11) -> adjudicatario
Lowest out-degree: ('CLARANET II SOLUTIONS, S.A.', 0) -> adjudicatario

Highest out-degree: ('Guarda Nacional Republicana', 66) -> adjudicante
Lowest in-degree: ('ACSS-Administração Central do Sistema de Saúde,IP', 0) -> adjudicante

Average in-degree: 0.92
Average out-degree: 0.92


The average in-degree and average out-degree are the same because, in any directed graph, the sum of all in-degrees equals the sum of all out-degrees, and both are equal to the number of edges.

Therefore:
  $$avg_{in}  = |E| / |V|$$
  $$avg_{out} = |E| / |V|$$

so they must be identical.

In [21]:
# Top 5 Degree Enitities
top5 = sorted(G.degree(), key=lambda x: x[1], reverse=True)[:5]
print("Top 5 most connected entities:")
for entity, deg in top5:
    print(f"  {entity}: {deg}")

Top 5 most connected entities:
  Guarda Nacional Republicana: 66
  Secretaria-Geral do Ministério da Administração Interna: 28
  Serviços Municipalizados de Água e Saneamento de Sintra: 19
  Casa Pia de Lisboa, I. P.: 18
  Instituto Nacional de Medicina Legal e Ciências Forenses, I. P.: 18


In [22]:
# Degree-1 entities
deg1 = sorted(
    [(node, degree) for node, degree in G.degree() if degree == 1],
    key=lambda x: x[0]
)

print(f"Degree-1 entities: {len(deg1)}")
for i, (entity, degree) in enumerate(deg1[:20], start=1):
    node_type = G.nodes[entity].get("node_type", "unknown")
    if node_type == "both":
        node_type = "adjudicante and adjudicatario"
    print(f"{i:>3}. {entity} (degree={degree}, type={node_type})")

if len(deg1) > 20:
    print(f"... and {len(deg1) - 20} more")

Degree-1 entities: 499
  1. - - Beatriz Soares Ribeiro (degree=1, type=adjudicatario)
  2. - - EBSCO Information Services S.L.U. (degree=1, type=adjudicatario)
  3. - - J. VILASECA, SA (degree=1, type=adjudicatario)
  4. - - J.VILASECA, SA (degree=1, type=adjudicatario)
  5. - - MARIO ANDRE GONÇALVES REIS (degree=1, type=adjudicatario)
  6. - - Mariana de Jesus Gonçalves Mamede Lopes (degree=1, type=adjudicatario)
  7. - - NOA GmbH (degree=1, type=adjudicatario)
  8. - - Onretrieval Group SL (degree=1, type=adjudicatario)
  9. - - Sérgio José Mamede Gonçalves (degree=1, type=adjudicatario)
 10. 1 - MEO - SERVIÇOS DE COMUNICAÇÕES E MULTIMÉDIA, S.A. (degree=1, type=adjudicatario)
 11. 2WayView Lda (degree=1, type=adjudicatario)
 12. : VECTORTRIBUTE UNIPESSOAL, LDA (degree=1, type=adjudicatario)
 13. A. Barreira, Lda. (degree=1, type=adjudicatario)
 14. A. Do Carmo , Importação , Exportação e Comercio, LDA (degree=1, type=adjudicatario)
 15. A. Gouv - Reparação e Manutenção de Veiculos, L

In [23]:
# All contract pairs with amounts
print("Contracts with their amounts:")
for u, v, data in G.edges(data=True):
    print(f"  {u} -- {v} : €{data['weight']:,}")

Contracts with their amounts:
  ACSS-Administração Central do Sistema de Saúde,IP -- CLARANET II SOLUTIONS, S.A. : €8.433811582477187
  ACSS-Administração Central do Sistema de Saúde,IP -- Primavera – Business Software Solutions S.A : €8.990615968707628
  ACSS-Administração Central do Sistema de Saúde,IP -- Timestamp - Sistemas de Informação, S.A. : €11.34954392823967
  ADENE - Agência para a Energia -- Digibéria Information Technologies S.A. : €11.923709734555993
  ADENE - Agência para a Energia -- Páginas aos Blocos, Lda. : €8.160932447399158
  ARTE - Agência para a Reforma Tecnológica do Estado -- MULTIMAC HITO INNOVATION, S.A. : €13.128303817996628
  AdP Valor - Serviços Ambientais, S. A. -- HCCM CONSULTING, S.A. : €13.283143542126385
  Administração Central do Sistema de Saúde, I. P. -- Claranet Portugal, S.A : €6.763908002983978
  Agrupamento de Escolas José Afonso, Loures -- Beltrão Coelho – Sistemas de Escritório, Lda : €13.064400840343946
  Agência Nacional para a Qualificação

## <font size=5>**3.4 Total Contracts Volume (Price) per Entity**</font> <a class="anchor" id="3.4"></a>
  
[Back to TOC](#toc)

In [24]:
volume = {node: 0 for node in G.nodes()}

# Step 2
for u, v, data in G.edges(data=True):
    volume[u] += data["weight"]
    volume[v] += data["weight"]

# Step 3
sorted_volume = sorted(volume.items(), key=lambda x: x[1], reverse=True)
print("Entity volume ranking:")
for entity, vol in sorted_volume:
    print(f"  {entity}: €{vol:,.0f}")

Entity volume ranking:
  Guarda Nacional Republicana: €646
  Secretaria-Geral do Ministério da Administração Interna: €303
  Serviços Municipalizados de Água e Saneamento de Sintra: €211
  Gebalis - Gestão do Arrendamento da Habitação Municipal de Lisboa, E. M., S. A.: €184
  Casa Pia de Lisboa, I. P.: €177
  Centro de Formação Profissional das Pescas e do Mar (FOR-MAR): €171
  Instituto Nacional de Medicina Legal e Ciências Forenses, I. P.: €158
  Instituto Superior de Economia e Gestão: €142
  Santa Casa da Misericórdia de Lisboa: €136
  Agência para a Modernização Administrativa, I. P.: €121
  Instituto de Ação Social das Forças Armadas, I. P.: €119
  Serviços Intermunicipalizados de Águas e Resíduos dos Municípios de Loures e Odivelas: €119
  Município de Mafra: €112
  Ministério da Defesa Nacional - Marinha: €111
  CLARANET II SOLUTIONS, S.A.: €109
  Agência Nacional para a Qualificação e o Ensino Profissional, I. P.: €100
  Município de Oeiras: €99
  ISEG - Instituto Superior de 

## <font size=5>**3.5 Degree VS Volume**</font> <a class="anchor" id="3.5"></a>
  
[Back to TOC](#toc)

- Degree $\rightarrow$ number of contracts associated with each entity (network degree = in_degree + out_degree)
- Volume $\rightarrow$ total contracted amount associated with each entity (sum of incident edge weights)

**Purpose**: assess which entities are most valuable and how activity (degree) relates to economic impact (volume).

Planned steps:
- compute degree and volume per node
- plot degree vs. volume (use log–log scatter), annotate top entities
- report Pearson and Spearman correlations
- fit a linear model on log-transformed values and flag outliers (high volume/low degree and high degree/low volume)

In [25]:
# Degree and volume per node
metrics = pd.DataFrame({
    "degree": pd.Series(dict(G.degree())),
    "volume": pd.Series(volume)
}).fillna(0)

In [26]:
# Keep only positive values for log-log analysis
pos = metrics[(metrics["degree"] > 0) & (metrics["volume"] > 0)].copy()
pos["log_degree"] = np.log10(pos["degree"])
pos["log_volume"] = np.log10(pos["volume"])

# Correlations on log-transformed values
pearson = pos["log_degree"].corr(pos["log_volume"], method="pearson")
spearman = pos["log_degree"].corr(pos["log_volume"], method="spearman")

print(f"Nodes used in log-log analysis: {len(pos)}")
print(f"Pearson correlation (log10):  {pearson:.4f}")
print(f"Spearman correlation (log10): {spearman:.4f}")

Nodes used in log-log analysis: 641
Pearson correlation (log10):  0.9593
Spearman correlation (log10): 0.7202


In [27]:
# Linear model in log-log space
slope, intercept = np.polyfit(pos["log_degree"], pos["log_volume"], 1)
pos["pred_log_volume"] = intercept + slope * pos["log_degree"]
pos["residual"] = pos["log_volume"] - pos["pred_log_volume"]

print(f"\nlog10(volume) = {intercept:.4f} + {slope:.4f} * log10(degree)")


log10(volume) = 1.0177 + 1.0032 * log10(degree)


In [28]:
# Flag outliers using the box-plot rule (IQR)
degree_dict = dict(G.degree())

q1 = pd.Series(volume).quantile(0.25)
q3 = pd.Series(volume).quantile(0.75)
iqr = q3 - q1
volume_threshold = q3 + 1.5 * iqr

print(f"Volume threshold from box plot rule: €{volume_threshold:,.0f}")
print("Flagged entities (degree ≥ 3 AND volume above upper fence):")

flagged = []
for entity in G.nodes():
    if degree_dict[entity] >= 3 and volume[entity] > volume_threshold:
        flagged.append((entity, degree_dict[entity], volume[entity]))

flagged.sort(key=lambda x: x[2], reverse=True)
for entity, deg, vol in flagged:
    print(f"  {entity}  degree={deg}  volume=€{vol:,.0f}")

print(f"\nTotal flagged: {len(flagged)}")

Volume threshold from box plot rule: €18
Flagged entities (degree ≥ 3 AND volume above upper fence):
  Guarda Nacional Republicana  degree=66  volume=€646
  Secretaria-Geral do Ministério da Administração Interna  degree=28  volume=€303
  Serviços Municipalizados de Água e Saneamento de Sintra  degree=19  volume=€211
  Gebalis - Gestão do Arrendamento da Habitação Municipal de Lisboa, E. M., S. A.  degree=17  volume=€184
  Casa Pia de Lisboa, I. P.  degree=18  volume=€177
  Centro de Formação Profissional das Pescas e do Mar (FOR-MAR)  degree=16  volume=€171
  Instituto Nacional de Medicina Legal e Ciências Forenses, I. P.  degree=18  volume=€158
  Instituto Superior de Economia e Gestão  degree=16  volume=€142
  Santa Casa da Misericórdia de Lisboa  degree=15  volume=€136
  Agência para a Modernização Administrativa, I. P.  degree=12  volume=€121
  Instituto de Ação Social das Forças Armadas, I. P.  degree=11  volume=€119
  Serviços Intermunicipalizados de Águas e Resíduos dos Municíp

In [29]:
# Plotting

# prepare fit line
x_line = np.logspace(np.log10(pos["degree"].min()), np.log10(pos["degree"].max()), 200)
y_line = 10 ** (intercept + slope * np.log10(x_line))

# main traces
scatter = go.Scatter(
    x=pos["degree"],
    y=pos["volume"],
    mode="markers",
    marker=dict(size=6, color="steelblue", opacity=0.5),
    name="nodes",
    hovertemplate="%{text}<br>Degree: %{x}<br>Volume: €%{y:,.0f}",
    text=pos.index
)

fit_line = go.Scatter(
    x=x_line,
    y=y_line,
    mode="lines",
    line=dict(color="crimson", width=2),
    name="log-log fit"
)

# annotations for top entities
annotations = []
top_entities = [entity for entity, _ in top5]

for node in top_entities:
    x = metrics.loc[node, "degree"]
    y = metrics.loc[node, "volume"]
    annotations.append(
        dict(
            x=x,
            y=y,
            text=node,
            showarrow=True,
            arrowhead=2,
            ax=10,
            ay=-10,
            font=dict(size=10),
        )
    )

fig = go.Figure(data=[scatter, fit_line])
fig.update_layout(
    title="Degree VS Volume",
    xaxis=dict(title="Degree", type="log"),
    yaxis=dict(title="Volume (€)", type="log"),
    annotations=annotations,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
    margin=dict(l=60, r=20, t=60, b=60)
)

fig.update_layout(
    title="Degree vs Volume",
    
    xaxis=dict(
        title="Degree",
        type="log",
        dtick=1,              # only 10^n ticks → 1, 10, 100, 1000
        tickformat=".0f"      # show full numbers instead of scientific notation
    ),
    
    yaxis=dict(
        title="Volume (€)",
        type="log",
        dtick=1,
        # tickformat=".0f"
    ),

    annotations=annotations,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
    margin=dict(l=60, r=20, t=60, b=60)
)

fig.update_xaxes(showgrid=True, gridwidth=1)
fig.update_yaxes(showgrid=True, gridwidth=1)

fig.show()

In [30]:
# TODO: add attribute adjudicante/adjudicatario to hoover in plot and maybe change the color of the point based on it 

**NETWORK MUNICIPALITY**

In [46]:
import pandas as pd
import numpy as np
import networkx as nx


# =========================================================
# 1. INPUT HANDLING (DICT / LIST / DF SAFE)
# =========================================================
def ensure_dataframe(data):

    if isinstance(data, pd.DataFrame):
        return data.copy()

    if isinstance(data, list):
        return pd.DataFrame(data)

    if isinstance(data, dict):
        # dict of dataframes
        for v in data.values():
            if isinstance(v, pd.DataFrame):
                return v.copy()

        # dict of records
        return pd.DataFrame(data)

    raise TypeError(f"Unsupported input type: {type(data)}")


# =========================================================
# 2. WEIGHT FUNCTION
# =========================================================
def compute_weight(x, mode="log_sum"):
    x = np.array(x)

    if mode == "log_sum":
        return np.log1p(x).sum()
    elif mode == "sum":
        return x.sum()
    elif mode == "mean":
        return x.mean()
    else:
        raise ValueError("Unknown weight mode")


def standardize_contract_data(df):

    df = df.copy()

    # =====================================================
    # 1. SOURCE (company)
    # =====================================================
    if 'source_company' in df.columns:
        pass

    elif 'adjudicante' in df.columns:
        df['source_company'] = df['adjudicante']

    elif 'source' in df.columns:
        df['source_company'] = df['source']

    elif 'adjudicatarios' in df.columns:
        # fallback case where structure is inverted/mixed
        df['source_company'] = df['adjudicante'] if 'adjudicante' in df.columns else np.nan

    else:
        raise ValueError(f"Cannot find source column. Columns: {df.columns.tolist()}")

    # =====================================================
    # 2. TARGET (company)
    # =====================================================
    if 'target_company' in df.columns:
        pass

    elif 'adjudicatarios' in df.columns:
        df['target_company'] = df['adjudicatarios']

    elif 'target' in df.columns:
        df['target_company'] = df['target']

    else:
        raise ValueError(f"Cannot find target column. Columns: {df.columns.tolist()}")

    # =====================================================
    # 3. PRICE
    # =====================================================
    if 'price' in df.columns:
        pass

    elif 'precoContratual' in df.columns:
        df['price'] = df['precoContratual']

    else:
        raise ValueError(f"Cannot find price column. Columns: {df.columns.tolist()}")

    # =====================================================
    # 4. CLEAN
    # =====================================================
    required = ['source_company', 'target_company', 'price']

    df = df.dropna(subset=required)
    df = df[df['price'] > 0]

    return df





# =========================================================
# 5. MUNICIPALITY NETWORK
# =========================================================
def build_municipality_network(data, weight_mode="log_sum"):

    df = ensure_dataframe(data)
    df = standardize_contract_data(df)

    if 'city' not in df.columns:
        raise ValueError("Missing 'city' column")

    df['source_municipality'] = df['city']
    df['target_company'] = df['target_company']

    edge_df = (
        df.groupby(['source_municipality', 'target_company'], as_index=False)
        .agg(
            total_price=('price', 'sum'),
            contracts=('price', 'count'),
            avg_price=('price', 'mean'),
            price_series=('price', list)
        )
    )

    edge_df['weight'] = edge_df['price_series'].apply(
        lambda x: compute_weight(x, mode=weight_mode)
    )

    G = nx.DiGraph()

    for row in edge_df.itertuples(index=False):
        G.add_edge(
            row.source_municipality,
            row.target_company,
            weight=row.weight,
            total_price=row.total_price,
            contracts=row.contracts,
            avg_price=row.avg_price
        )

    return G


# =========================================================
# 6. VISUAL ATTRIBUTES (OPTIONAL)
# =========================================================
def add_visual_attributes(
    G,
    thickness_mode="linear",
    size_mode="linear",
    scale_min=1,
    scale_max=5,
    suffix=""
):

    conc = np.array([d.get('nr_concorrentes', 0) for _, _, d in G.edges(data=True)])
    weight = np.array([d['weight'] for _, _, d in G.edges(data=True)])

    def normalize(x):
        if len(x) == 0:
            return x

        r = np.ptp(x)
        if r == 0:
            return np.zeros_like(x)

        return (x - np.min(x)) / (r + 1e-9)

    if thickness_mode == "log":
        conc = np.log1p(conc)

    if size_mode == "log":
        weight = np.log1p(weight)

    conc_norm = normalize(conc)
    weight_norm = normalize(weight)

    def scale(x):
        return scale_min + (scale_max - scale_min) * x

    for i, (_, _, d) in enumerate(G.edges(data=True)):
        d[f'edge_thickness{suffix}'] = scale(conc_norm[i])
        d[f'edge_size{suffix}'] = scale(weight_norm[i])

    return G


# =========================================================
# 7. PIPELINE USAGE
# =========================================================

df = data  # raw input (dict/list/DataFrame supported)

# MUNICIPALITY NETWORK
G_municipality = build_municipality_network(df)
G_municipality = add_visual_attributes(G_municipality)

ValueError: Cannot find source column. Columns: ['weight', 'total_price', 'nr_concorrentes', 'contracts', 'weight_mode', 'idcontrato', 'tipoContrato', 'tipoFimContrato', 'CPV', 'precoBaseProcedimento', 'precoContratual', 'PrecoTotalEfetivo', 'dataDecisaoAdjudicacao', 'dataCelebracaoContrato', 'dataPublicacao', 'dataFechoContrato', 'nr_concorrentes_list', 'contribuinte_adjudicante', 'contribuinte_adjudicatarios', 'city', 'cpv_prefix', 'agg_cpv']

In [40]:
df = _ensure_dataframe(data)
print(df.columns)

Index(['weight', 'total_price', 'nr_concorrentes', 'contracts', 'weight_mode',
       'idcontrato', 'tipoContrato', 'tipoFimContrato', 'CPV',
       'precoBaseProcedimento', 'precoContratual', 'PrecoTotalEfetivo',
       'dataDecisaoAdjudicacao', 'dataCelebracaoContrato', 'dataPublicacao',
       'dataFechoContrato', 'nr_concorrentes_list', 'contribuinte_adjudicante',
       'contribuinte_adjudicatarios', 'city', 'cpv_prefix', 'agg_cpv'],
      dtype='str')


## <font size=5>**3.6 Exporting It**</font> <a class="anchor" id="3.6"></a>
  
[Back to TOC](#toc)

In [31]:
output_path = '../graphs/gephi_graph01.gexf'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
nx.write_gexf(G, output_path)
print(f'Graph exported to {output_path}')

TypeError: cannot unpack non-iterable int object